# Trabalho prático I — RNA e Lógica Fuzzy

Este notebook contém **somente a preparação dos dados**. Ao final, estarão disponíveis:


## 1. Bibliotecas

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

SEMENTE = 42
np.random.seed(SEMENTE)

## 2. Carregamento da base

Coloque o arquivo `dados_sinteticos_neuro_fuzzy_10000.xlsx` na mesma pasta do notebook.

In [ ]:
NOME_ARQUIVO = "dados_sinteticos_neuro_fuzzy_10000.xlsx"

candidatos = [
    Path(NOME_ARQUIVO),
    Path("upload") / NOME_ARQUIVO,
]
caminho = next((p for p in candidatos if p.exists()), None)

if caminho is None:
    try:
        from google.colab import files

        print("Selecione a planilha fornecida pela professora.")
        arquivos = files.upload()
        caminho = Path(next(iter(arquivos)))
    except ImportError as erro:
        raise FileNotFoundError(
            f"Arquivo não encontrado. Coloque '{NOME_ARQUIVO}' na mesma pasta do notebook."
        ) from erro

dados = pd.read_excel(caminho, sheet_name="Dados")
print(f"Arquivo carregado: {caminho}")
print(f"Dimensões da base: {dados.shape[0]} linhas e {dados.shape[1]} colunas")
dados.head()

## 3. Conferência inicial

As células abaixo permitem verificar os tipos das colunas, os valores ausentes e a distribuição das classes.

In [ ]:
resumo_colunas = pd.DataFrame(
    {
        "tipo": dados.dtypes.astype(str),
        "ausentes": dados.isna().sum(),
        "valores_unicos": dados.nunique(dropna=True),
    }
)
resumo_colunas

In [ ]:
distribuicao_classes = dados["classe"].value_counts().sort_index()
display(distribuicao_classes.to_frame("quantidade"))

distribuicao_classes.plot(
    kind="bar", color="#2F6B5F", edgecolor="black", figsize=(7, 4)
)
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade de registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Separação das variáveis

Serão utilizadas como entradas as seis medições e os três alarmes descritos no trabalho. As colunas `id`, `cenario` e `sensor_ausente` serão preservadas apenas para identificar e analisar posteriormente os resultados; `classe` será a variável-alvo. Assim, informações sobre a forma como o exemplo foi produzido não entram no modelo.

In [ ]:
COLUNAS_NUMERICAS = [
    "temperatura_c",
    "latencia_ms",
    "perda_pacotes_pct",
    "tensao_v",
    "rotacao_rpm",
    "reinicializacoes_h",
]

COLUNAS_BINARIAS = [
    "alarme_rede",
    "alarme_temperatura",
    "alarme_tensao",
]

COLUNAS_ENTRADA = COLUNAS_NUMERICAS + COLUNAS_BINARIAS
COLUNAS_AUXILIARES = ["id", "cenario", "sensor_ausente"]

X = dados[COLUNAS_ENTRADA].copy()
y = dados["classe"].copy()
auxiliares = dados[COLUNAS_AUXILIARES].copy()

print("Variáveis de entrada:", COLUNAS_ENTRADA)
print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

## 5. Divisão em treino e teste

A divisão abaixo reserva 80% dos registros para treino e 20% para teste. O parâmetro `stratify=y` mantém a proporção das quatro classes nos dois conjuntos. Os índices também são divididos para que `id`, `cenario` e `sensor_ausente` possam ser usados na análise dos erros sem participar da aprendizagem.

In [ ]:
indices = np.arange(len(dados))

idx_treino, idx_teste = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEMENTE,
    stratify=y,
)

X_treino = X.iloc[idx_treino].copy()
X_teste = X.iloc[idx_teste].copy()
y_treino = y.iloc[idx_treino].copy()
y_teste = y.iloc[idx_teste].copy()
aux_treino = auxiliares.iloc[idx_treino].copy()
aux_teste = auxiliares.iloc[idx_teste].copy()

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)
print("\nProporção das classes no treino:")
display(
    y_treino.value_counts(normalize=True).sort_index().to_frame("proporção")
)
print("Proporção das classes no teste:")
display(
    y_teste.value_counts(normalize=True).sort_index().to_frame("proporção")
)

## 6. Preparação para a RNA

Nos dados numéricos, os valores ausentes serão substituídos pela mediana calculada **somente no treino**. Em seguida, essas variáveis serão padronizadas. As colunas binárias já estão na escala 0–1 e serão apenas copiadas. O mesmo transformador ajustado no treino será aplicado ao teste, evitando vazamento de dados.

In [ ]:
pipeline_numerico = Pipeline(
    steps=[
        ("imputacao", SimpleImputer(strategy="median")),
        ("padronizacao", StandardScaler()),
    ]
)

preparador_rna = ColumnTransformer(
    transformers=[
        ("numericas", pipeline_numerico, COLUNAS_NUMERICAS),
        ("binarias", "passthrough", COLUNAS_BINARIAS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_treino_rna = preparador_rna.fit_transform(X_treino)
X_teste_rna = preparador_rna.transform(X_teste)

codificador_classes = LabelEncoder()
y_treino_cod = codificador_classes.fit_transform(y_treino)
y_teste_cod = codificador_classes.transform(y_teste)

nomes_entradas_rna = preparador_rna.get_feature_names_out().tolist()
nomes_classes = codificador_classes.classes_.tolist()

print("Formato dos dados da RNA:")
print("X_treino_rna:", X_treino_rna.shape)
print("X_teste_rna: ", X_teste_rna.shape)
print("Ordem das classes:", dict(enumerate(nomes_classes)))

In [ ]:
# Verificações simples da preparação
assert X_treino_rna.shape == (8000, 9)
assert X_teste_rna.shape == (2000, 9)
assert not np.isnan(X_treino_rna).any()
assert not np.isnan(X_teste_rna).any()
assert len(nomes_classes) == 4
print("Sem valores ausentes.")

### A partir daqui, a equipe deverá construir a RNA

Utilize `X_treino_rna` e `y_treino_cod` para o treinamento. Use `X_teste_rna` e `y_teste_cod` apenas na avaliação final. A arquitetura solicitada possui duas camadas ocultas, com 32 e 16 neurônios, ativação ReLU e quatro saídas com softmax.

In [ ]:
# PARTE DOS ALUNOS: construir, treinar e avaliar a RNA aqui.
# Apresentar acurácia, precisão, revocação, macro-F1 e matriz de confusão.

# Seu código:

## 7. Preparação para o sistema fuzzy

Para o sistema fuzzy, as variáveis devem continuar em suas unidades originais, pois os limites das funções de pertinência serão definidos em °C, ms, %, V, rpm e quantidade de reinicializações. Por isso, será feita somente a imputação pela mediana do treino, sem padronização.

In [ ]:
imputador_fuzzy = SimpleImputer(strategy="median")

X_treino_fuzzy = X_treino.copy()
X_teste_fuzzy = X_teste.copy()

X_treino_fuzzy[COLUNAS_NUMERICAS] = imputador_fuzzy.fit_transform(
    X_treino[COLUNAS_NUMERICAS]
)
X_teste_fuzzy[COLUNAS_NUMERICAS] = imputador_fuzzy.transform(
    X_teste[COLUNAS_NUMERICAS]
)

assert not X_treino_fuzzy.isna().any().any()
assert not X_teste_fuzzy.isna().any().any()

print("Dados fuzzy preparados nas unidades originais.")
X_treino_fuzzy.head()

### A partir daqui, a equipe deverá construir o sistema fuzzy

A equipe deverá:

1. definir e representar graficamente as funções de pertinência;
2. construir os módulos de rede, refrigeração e alimentação;
3. elaborar pelo menos quatro regras para cada módulo;

#### 1. Funções de pertinência e fuzzificação

Os limites abaixo partem das faixas sugeridas na Tabela 3 do enunciado e foram conferidos contra os valores observados na base. Funções trapezoidais são usadas nos extremos para representar regiões estáveis e funções triangulares representam os estados intermediários. As sobreposições são intencionais: um valor de fronteira pode pertencer parcialmente a dois termos linguísticos, evitando mudanças bruscas no diagnóstico.

Além dos gráficos, a célula transforma cada uma das seis medições em três graus de pertinência no intervalo $[0, 1]$. Assim, cada registro passa a possuir 18 características fuzzy. Os alarmes binários permanecem disponíveis como evidências adicionais para as regras dos módulos, conforme permitido pelo enunciado.

In [ ]:
# Funções de pertinência implementadas diretamente com NumPy.
# Isso deixa explícito como cada grau é calculado e evita dependências adicionais.
def pertinencia_triangular(x, a, b, c):
    """Calcula a função de pertinência triangular definida por (a, b, c)."""
    if not a < b < c:
        raise ValueError("Os parâmetros triangulares devem satisfazer a < b < c.")

    x = np.asarray(x, dtype=float)
    subida = (x - a) / (b - a)
    descida = (c - x) / (c - b)
    return np.clip(np.minimum(subida, descida), 0.0, 1.0)


def pertinencia_trapezoidal(x, a, b, c, d):
    """Calcula a função trapezoidal, incluindo ombros com a=b ou c=d."""
    if not a <= b <= c <= d or a == d:
        raise ValueError("Os parâmetros trapezoidais devem satisfazer a <= b <= c <= d.")

    x = np.asarray(x, dtype=float)
    graus = np.zeros_like(x)

    if a == b:  # ombro esquerdo
        graus[x <= b] = 1.0
    else:
        mascara_subida = (x > a) & (x < b)
        graus[mascara_subida] = (x[mascara_subida] - a) / (b - a)

    graus[(x >= b) & (x <= c)] = 1.0

    if c == d:  # ombro direito
        graus[x >= c] = 1.0
    else:
        mascara_descida = (x > c) & (x < d)
        graus[mascara_descida] = (d - x[mascara_descida]) / (d - c)

    return np.clip(graus, 0.0, 1.0)


# Os limites partem da Tabela 3 do enunciado e respeitam as faixas da base.
# Formato de cada termo: (tipo da função, parâmetros numéricos).
CONFIG_PERTINENCIA = {
    "temperatura_c": {
        "rotulo": "Temperatura", "unidade": "°C", "universo": (20, 100),
        "termos": {
            "normal": ("trap", (20, 20, 45, 55)),
            "alta": ("tri", (45, 65, 80)),
            "critica": ("trap", (70, 80, 100, 100)),
        },
    },
    "latencia_ms": {
        "rotulo": "Latência", "unidade": "ms", "universo": (1, 500),
        "termos": {
            "baixa": ("trap", (1, 1, 30, 60)),
            "moderada": ("tri", (40, 100, 160)),
            "alta": ("trap", (120, 200, 500, 500)),
        },
    },
    "perda_pacotes_pct": {
        "rotulo": "Perda de pacotes", "unidade": "%", "universo": (0, 100),
        "termos": {
            "pequena": ("trap", (0, 0, 2, 5)),
            "media": ("tri", (2, 12, 25)),
            "severa": ("trap", (15, 30, 100, 100)),
        },
    },
    "tensao_v": {
        "rotulo": "Tensão", "unidade": "V", "universo": (180, 250),
        "termos": {
            "baixa": ("trap", (180, 180, 200, 210)),
            "adequada": ("trap", (205, 212, 228, 235)),
            "alta": ("trap", (230, 240, 250, 250)),
        },
    },
    "rotacao_rpm": {
        "rotulo": "Rotação do ventilador", "unidade": "rpm", "universo": (0, 5000),
        "termos": {
            "baixa": ("trap", (0, 0, 900, 1300)),
            "media": ("tri", (900, 1750, 2600)),
            "adequada": ("trap", (2200, 3000, 5000, 5000)),
        },
    },
    "reinicializacoes_h": {
        "rotulo": "Reinicializações por hora", "unidade": "/h", "universo": (0, 10),
        "termos": {
            "rara": ("trap", (0, 0, 0.5, 1.5)),
            "moderada": ("tri", (1, 2.5, 4)),
            "frequente": ("trap", (3, 5, 10, 10)),
        },
    },
    "grau_falha": {
        "rotulo": "Grau de falha", "unidade": "0–100", "universo": (0, 100),
        "termos": {
            "baixo": ("trap", (0, 0, 25, 40)),
            "medio": ("tri", (25, 50, 75)),
            "alto": ("trap", (60, 75, 100, 100)),
        },
    },
}


def calcular_pertinencia(valores, tipo, parametros):
    if tipo == "tri":
        return pertinencia_triangular(valores, *parametros)
    if tipo == "trap":
        return pertinencia_trapezoidal(valores, *parametros)
    raise ValueError(f"Tipo de função desconhecido: {tipo}")


# Tabela auditável com todos os parâmetros adotados.
linhas_parametros = []
for variavel, configuracao in CONFIG_PERTINENCIA.items():
    for termo, (tipo, parametros) in configuracao["termos"].items():
        linhas_parametros.append({
            "variavel": variavel,
            "termo": termo,
            "funcao": "triangular" if tipo == "tri" else "trapezoidal",
            "parametros": parametros,
        })

tabela_parametros_pertinencia = pd.DataFrame(linhas_parametros)
display(tabela_parametros_pertinencia)


# Representação gráfica das seis entradas e da saída usada pelos módulos.
fig, eixos = plt.subplots(4, 2, figsize=(15, 18))
eixos = eixos.ravel()
cores = ["#2F6B5F", "#E69F00", "#C44E52"]

for eixo, (variavel, configuracao) in zip(eixos, CONFIG_PERTINENCIA.items()):
    inicio, fim = configuracao["universo"]
    universo = np.linspace(inicio, fim, 1001)

    for cor, (termo, (tipo, parametros)) in zip(cores, configuracao["termos"].items()):
        graus = calcular_pertinencia(universo, tipo, parametros)
        eixo.plot(universo, graus, linewidth=2, color=cor, label=termo)

    eixo.set_title(configuracao["rotulo"])
    eixo.set_xlabel(f"{configuracao['rotulo']} ({configuracao['unidade']})")
    eixo.set_ylabel("Grau de pertinência μ(x)")
    eixo.set_xlim(inicio, fim)
    eixo.set_ylim(-0.03, 1.05)
    eixo.grid(alpha=0.25)
    eixo.legend()

for eixo in eixos[len(CONFIG_PERTINENCIA):]:
    eixo.axis("off")

fig.suptitle("Funções de pertinência do sistema fuzzy", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


# Fuzzificação efetiva dos dados: cada medição vira três graus entre 0 e 1.
def fuzzificar_dataframe(dados_entrada):
    resultado = pd.DataFrame(index=dados_entrada.index)

    for variavel in COLUNAS_NUMERICAS:
        configuracao = CONFIG_PERTINENCIA[variavel]
        valores = dados_entrada[variavel].to_numpy(dtype=float)
        graus_da_variavel = []

        for termo, (tipo, parametros) in configuracao["termos"].items():
            graus = calcular_pertinencia(valores, tipo, parametros)
            resultado[f"{variavel}__{termo}"] = graus
            graus_da_variavel.append(graus)

        # Toda medição válida deve ativar pelo menos um termo linguístico.
        maior_ativacao = np.max(np.column_stack(graus_da_variavel), axis=1)
        if np.any(maior_ativacao <= 0):
            raise ValueError(f"Há valores de {variavel} sem cobertura fuzzy.")

    return resultado


X_treino_fuzzificado = fuzzificar_dataframe(X_treino_fuzzy)
X_teste_fuzzificado = fuzzificar_dataframe(X_teste_fuzzy)

assert X_treino_fuzzificado.shape == (8000, 18)
assert X_teste_fuzzificado.shape == (2000, 18)
assert X_treino_fuzzificado.to_numpy().min() >= 0
assert X_treino_fuzzificado.to_numpy().max() <= 1
assert X_teste_fuzzificado.to_numpy().min() >= 0
assert X_teste_fuzzificado.to_numpy().max() <= 1

print("Treino fuzzificado:", X_treino_fuzzificado.shape)
print("Teste fuzzificado: ", X_teste_fuzzificado.shape)
display(pd.concat([
    X_treino_fuzzy[COLUNAS_NUMERICAS].head(3),
    X_treino_fuzzificado.head(3),
], axis=1))

In [ ]:
# PARTE DOS ALUNOS: criar as regras dos três módulos fuzzy.

# Seu código:

In [ ]:
# PARTE DOS ALUNOS: demonstrar fuzzificação, inferência e defuzzificação.

# Seu código:

## 8. Objetos disponíveis para o trabalho

| Objeto | Uso |
|---|---|
| `X_treino_rna` | Entradas padronizadas para treinar a RNA |
| `X_teste_rna` | Entradas padronizadas para testar a RNA |
| `y_treino_cod` | Classes numéricas do treino |
| `y_teste_cod` | Classes numéricas do teste |
| `nomes_classes` | Ordem usada na codificação das classes |
| `X_treino_fuzzy` | Entradas de treino nas unidades originais |
| `X_teste_fuzzy` | Entradas de teste nas unidades originais |
| `y_treino`, `y_teste` | Classes em texto |
| `aux_treino`, `aux_teste` | ID, cenário e indicação de sensor ausente para análises |
| `preparador_rna` | Transformador ajustado no treino para preparar novos dados |
| `imputador_fuzzy` | Imputador ajustado no treino para o sistema fuzzy |